# LOFO novelty launch

The default is a CPU-safe dry run. Set `DRY_RUN = False` after confirming the
environment cell and safe embedded split metadata. The full run trains two
families per GPU using `CUDA_VISIBLE_DEVICES`, then writes `lofo_{family}.json`
and `novelty_lofo.json` to `/kaggle/working`.


In [ ]:
import sys
print('python', sys.version)
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpu_count', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print('gpu', i, torch.cuda.get_device_name(i))
except Exception as e:
    print('torch unavailable:', type(e).__name__, e)
try:
    import timm
    print('timm', timm.__version__)
except Exception as e:
    print('timm unavailable:', type(e).__name__, e)
print('script_sha256_expected', 'bf5041fa43ea8ed25eccfbc8e1aecaa5e3b1ee07f90ddd97b7b0570716fa4e43')
print('locked_rows_excluded_from_embedded_splits', 1457)


In [ ]:
from pathlib import Path
import base64, io, zlib
import pandas as pd

encoded = '...'
frozen_text = zlib.decompress(base64.b64decode(encoded)).decode('utf-8')
Path('/kaggle/working/splits.csv').write_text(frozen_text, encoding='utf-8')
frozen = pd.read_csv(io.StringIO(frozen_text))
split = frozen['split'].astype(str).str.lower()
assert not split.isin(['holdout', 'locked_holdout', 'locked']).any(), 'locked rows in embedded splits'
print('wrote /kaggle/working/splits.csv rows', len(frozen), 'columns', list(frozen.columns))
for candidate in Path('/kaggle/input').rglob('splits.csv'):
    attached = pd.read_csv(candidate)
    attached_split = attached['split'].astype(str).str.lower()
    attached = attached.loc[~attached_split.isin(['holdout', 'locked_holdout', 'locked'])].copy()
    if frozen.equals(attached):
        print('attached safe splits.csv matches embedded metadata:', candidate)
    else:
        print('attached splits.csv differs; using embedded safe metadata:', candidate)


In [ ]:
from pathlib import Path
SOURCE = '"""Leave-one-family-out novelty evaluation for the QualityTwin image data.\n\nThe script is intentionally self-contained so it can be embedded in a Kaggle\nnotebook.  It trains one EfficientNet-B0 per held-out family on fold-0\'s\nknown-class training rows, then evaluates max-softmax, energy, and kNN cosine\ndistance novelty scores on fold-0 validation plus ``cond_holdout`` rows.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\nfrom typing import Any, Sequence\n\nimport numpy as np\n\nDATA_ROOT = os.environ.get("QUALITYTWIN_DATA_ROOT", "")\nSPLITS_CSV = os.environ.get("QUALITYTWIN_SPLITS_CSV", "")\nOUT = os.environ.get("QUALITYTWIN_OUT", "/kaggle/working")\nEPOCHS = 8\nIMG = 256\nSEED = 0\nGPU = os.environ.get("QUALITYTWIN_GPU", "auto")\nBATCH_SIZE = 64\nLR = 3e-4\nWEIGHT_DECAY = 1e-2\nFAMILIES = ["crack", "hole", "rust", "scratch"]\nARCH = "efficientnet_b0"\nLOCKED_WORDS = {"holdout", "locked", "locked_holdout", "locked-holdout"}\n\n\ndef script_sha256() -> str:\n    try:\n        return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    except OSError:\n        return hashlib.sha256(b"embedded-lofo_novelty").hexdigest()\n\n\ndef version_of(name: str) -> str:\n    try:\n        module = __import__(name)\n        return str(getattr(module, "__version__", "unknown"))\n    except Exception as exc:\n        return f"unavailable ({type(exc).__name__})"\n\n\ndef print_banner(device: str) -> None:\n    print("=== QualityTwin lofo_novelty startup ===", flush=True)\n    print(f"python={platform.python_version()} torch={version_of(\'torch\')} timm={version_of(\'timm\')}", flush=True)\n    try:\n        import torch\n        print(f"gpu_count={torch.cuda.device_count()} cuda_available={torch.cuda.is_available()} device={device}", flush=True)\n        for i in range(torch.cuda.device_count()):\n            print(f"gpu[{i}]={torch.cuda.get_device_name(i)}", flush=True)\n    except Exception as exc:\n        print(f"gpu_count=unavailable ({type(exc).__name__}) device={device}", flush=True)\n    print(f"script_sha256={script_sha256()}", flush=True)\n    print("data_policy=dev fold-0/other-dev folds + cond_holdout only; locked holdout excluded", flush=True)\n\n\ndef parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data-root", default=DATA_ROOT)\n    parser.add_argument("--splits-csv", default=SPLITS_CSV)\n    parser.add_argument("--out", default=OUT)\n    parser.add_argument("--families", default=",".join(FAMILIES), help="comma-separated held-out families")\n    parser.add_argument("--gpu", default=GPU)\n    parser.add_argument("--epochs", type=int, default=EPOCHS)\n    parser.add_argument("--img", type=int, default=IMG)\n    parser.add_argument("--seed", type=int, default=SEED)\n    parser.add_argument("--batch-size", type=int, default=BATCH_SIZE)\n    parser.add_argument("--dry-run", action="store_true")\n    parser.add_argument("--launch-two-gpus", action="store_true")\n    return parser.parse_args(argv)\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n    except Exception:\n        pass\n\n\ndef row_is_locked(row: dict[str, str]) -> bool:\n    for value in row.values():\n        text = str(value).strip().lower()\n        if text in LOCKED_WORDS or ("locked" in text and "cond" not in text):\n            return True\n    return str(row.get("split", "")).strip().lower() in {"holdout", "locked_holdout"}\n\n\ndef find_splits_csv(explicit: str) -> Path:\n    candidates: list[Path] = []\n    if explicit:\n        candidates.append(Path(explicit))\n    for base in (Path("/kaggle/input"), Path.cwd(), Path(__file__).resolve().parents[1]):\n        if base.exists():\n            candidates.extend(sorted(base.rglob("splits.csv")))\n    for path in candidates:\n        if path.is_file():\n            return path.resolve()\n    raise FileNotFoundError("Could not find splits.csv; pass --splits-csv or attach the dataset")\n\n\ndef read_metadata(path: Path) -> Any:\n    """Read only safe metadata rows; locked rows never enter the DataFrame."""\n    import pandas as pd\n\n    safe_rows: list[dict[str, str]] = []\n    locked_count = 0\n    with path.open("r", encoding="utf-8", newline="") as handle:\n        reader = csv.DictReader(handle)\n        fields = list(reader.fieldnames or [])\n        for row in reader:\n            if row_is_locked(row):\n                locked_count += 1\n                continue\n            safe_rows.append(row)\n    required = {"file", "cls", "split", "fold"}\n    missing = required.difference(fields)\n    if missing:\n        raise ValueError(f"splits.csv missing columns {sorted(missing)}; columns={fields}")\n    df = pd.DataFrame.from_records(safe_rows, columns=fields)\n    assert not bool(df.apply(lambda row: row_is_locked(row.to_dict()), axis=1).any()), "locked-holdout rows reached the trainer"\n    assert not bool(df["split"].astype(str).str.strip().str.lower().isin({"holdout", "locked_holdout"}).any()), "locked holdout rows reached the trainer"\n    print(f"splits columns={fields} rows_after_locked_filter={len(df)} dropped_locked={locked_count}", flush=True)\n    return df.reset_index(drop=True)\n\n\ndef find_data_root(explicit: str, split_path: Path, first_file: str) -> Path:\n    if explicit:\n        root = Path(explicit)\n        if (root / first_file).exists():\n            return root.resolve()\n        raise FileNotFoundError(f"--data-root does not contain {first_file}: {root}")\n    search_roots = [Path("/kaggle/input"), split_path.parent, Path.cwd(), Path(__file__).resolve().parents[1]]\n    for base in search_roots:\n        if not base.exists():\n            continue\n        if (base / first_file).exists():\n            return base.resolve()\n        for path in base.rglob(Path(first_file).name):\n            candidate = path\n            for _ in Path(first_file).parts:\n                candidate = candidate.parent\n            if (candidate / first_file).exists():\n                return candidate.resolve()\n    raise FileNotFoundError(f"Could not locate image {first_file}; pass --data-root")\n\n\ndef load_images(df: Any, root: Path, img: int) -> np.ndarray:\n    from concurrent.futures import ThreadPoolExecutor\n    from PIL import Image\n\n    rs = getattr(Image, "Resampling", Image).BILINEAR\n\n    def load_one(rel: str) -> np.ndarray:\n        with Image.open(root / str(rel)) as image:\n            return np.asarray(image.convert("L").resize((img, img), rs), dtype=np.uint8)\n\n    t0 = time.perf_counter()\n    workers = min(8, max(1, os.cpu_count() or 4))\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        images = np.stack(list(pool.map(load_one, df["file"].tolist())))\n    print(f"preloaded {len(images)} images shape={images.shape} seconds={time.perf_counter()-t0:.2f}", flush=True)\n    return images\n\n\ndef make_input(torch: Any, x_u8: np.ndarray, device: Any) -> Any:\n    x = torch.from_numpy(np.ascontiguousarray(x_u8)).to(device=device, dtype=torch.float32).div_(255.0)\n    x = x.unsqueeze(1).repeat(1, 3, 1, 1)\n    mean = torch.tensor([0.449, 0.449, 0.449], device=device).view(1, 3, 1, 1)\n    std = torch.tensor([0.226, 0.226, 0.226], device=device).view(1, 3, 1, 1)\n    return (x - mean) / std\n\n\ndef _affine(torch: Any, x: Any, angles: Any, scales: Any) -> Any:\n    n = x.shape[0]\n    rad = angles * math.pi / 180.0\n    c, s = torch.cos(rad) / scales, torch.sin(rad) / scales\n    theta = torch.zeros((n, 2, 3), device=x.device, dtype=x.dtype)\n    theta[:, 0, 0], theta[:, 0, 1] = c, s\n    theta[:, 1, 0], theta[:, 1, 1] = -s, c\n    grid = torch.nn.functional.affine_grid(theta, x.size(), align_corners=False)\n    return torch.nn.functional.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)\n\n\ndef augment_gpu(torch: Any, x: Any, seed: int, img: int) -> Any:\n    g = torch.Generator(device=x.device)\n    g.manual_seed(int(seed) & 0x7FFFFFFF)\n    n = x.shape[0]\n    active = torch.rand(n, device=x.device, generator=g) >= 0.40\n    if not bool(active.any()):\n        return x\n    z = x.clone()\n    idx = active.nonzero(as_tuple=False).flatten()\n    a = z[idx]\n    choose = torch.rand(len(idx), device=x.device, generator=g)\n    rot90 = choose < 0.16\n    if bool(rot90.any()):\n        a[rot90] = torch.rot90(a[rot90], 1, (-2, -1))\n    flip_h = torch.rand(len(idx), device=x.device, generator=g) < 0.25\n    flip_v = torch.rand(len(idx), device=x.device, generator=g) < 0.25\n    if bool(flip_h.any()):\n        a[flip_h] = a[flip_h].flip(-1)\n    if bool(flip_v.any()):\n        a[flip_v] = a[flip_v].flip(-2)\n    rotate = torch.rand(len(idx), device=x.device, generator=g) < 0.20\n    if bool(rotate.any()):\n        q = rotate.nonzero(as_tuple=False).flatten()\n        angles = torch.empty(len(q), device=x.device).uniform_(-180, 180, generator=g)\n        a[q] = _affine(torch, a[q], angles, torch.ones(len(q), device=x.device))\n    crop = torch.rand(len(idx), device=x.device, generator=g) < 0.18\n    if bool(crop.any()):\n        q = crop.nonzero(as_tuple=False).flatten()\n        scale = torch.empty(len(q), device=x.device).uniform_(0.6, 1.0, generator=g)\n        a[q] = _affine(torch, a[q], torch.zeros(len(q), device=x.device), scale)\n    b = torch.empty(len(idx), 1, 1, 1, device=x.device).uniform_(0.8, 1.2, generator=g)\n    c = torch.empty(len(idx), 1, 1, 1, device=x.device).uniform_(0.8, 1.2, generator=g)\n    gamma = torch.empty(len(idx), 1, 1, 1, device=x.device).uniform_(0.75, 1.35, generator=g)\n    mean = a.mean((-2, -1), keepdim=True)\n    a = ((a - mean) * c + mean) * b\n    a = a.clamp(0, 1).pow(gamma)\n    noise = torch.empty_like(a).normal_(0, 0.035, generator=g)\n    a = torch.where(torch.rand(len(idx), 1, 1, 1, device=x.device, generator=g) < 0.28, a + noise, a)\n    small = torch.rand(len(idx), device=x.device, generator=g) < 0.18\n    if bool(small.any()):\n        q = torch.nn.functional.interpolate(a[small], size=(max(8, img // 4), max(8, img // 4)), mode="area")\n        a[small] = torch.nn.functional.interpolate(q, size=(img, img), mode="bilinear", align_corners=False)\n    z[idx] = a.clamp(0, 1)\n    return z\n\n\ndef amp_context(torch: Any, device: Any):\n    return torch.autocast("cuda", dtype=torch.float16) if getattr(device, "type", str(device)) == "cuda" else torch.autocast("cpu", enabled=False)\n\n\ndef make_model(timm: Any, device: Any, n_classes: int, pretrained: bool) -> Any:\n    try:\n        model = timm.create_model(ARCH, pretrained=pretrained, num_classes=n_classes)\n    except Exception:\n        if not pretrained:\n            raise\n        print("pretrained weights unavailable; retrying without download", flush=True)\n        model = timm.create_model(ARCH, pretrained=False, num_classes=n_classes)\n    return model.to(device)\n\n\ndef forward_with_embedding(model: Any, x: Any) -> tuple[Any, Any]:\n    features = model.forward_features(x)\n    embedding = model.forward_head(features, pre_logits=True)\n    if embedding.ndim > 2:\n        embedding = embedding.flatten(1)\n    logits = model.get_classifier()(embedding)\n    return logits, embedding\n\n\ndef predict(model: Any, X: np.ndarray, indices: np.ndarray, device: Any, batch_size: int, torch: Any) -> tuple[np.ndarray, np.ndarray]:\n    logits_parts: list[np.ndarray] = []\n    emb_parts: list[np.ndarray] = []\n    model.eval()\n    with torch.inference_mode():\n        for start in range(0, len(indices), batch_size):\n            xb = make_input(torch, X[indices[start:start + batch_size]], device)\n            with amp_context(torch, device):\n                logits, emb = forward_with_embedding(model, xb)\n            logits_parts.append(logits.float().cpu().numpy())\n            emb_parts.append(emb.float().cpu().numpy())\n    return np.concatenate(logits_parts), np.concatenate(emb_parts)\n\n\ndef normalize_rows(x: np.ndarray) -> np.ndarray:\n    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)\n\n\ndef auroc(y: np.ndarray, scores: np.ndarray) -> float:\n    order = np.argsort(scores, kind="mergesort")\n    ranks = np.empty(len(scores), dtype=np.float64)\n    sorted_scores = scores[order]\n    i = 0\n    while i < len(scores):\n        j = i + 1\n        while j < len(scores) and sorted_scores[j] == sorted_scores[i]:\n            j += 1\n        ranks[order[i:j]] = (i + j - 1) / 2.0 + 1.0\n        i = j\n    pos = float(np.sum(y == 1))\n    neg = float(np.sum(y == 0))\n    if not pos or not neg:\n        return float("nan")\n    return float((ranks[y == 1].sum() - pos * (pos + 1) / 2.0) / (pos * neg))\n\n\ndef threshold_at_fpr(scores: np.ndarray, y: np.ndarray, target: float = 0.05) -> tuple[float, float, float]:\n    known = np.asarray(scores[y == 0], dtype=np.float64)\n    positives = np.asarray(scores[y == 1], dtype=np.float64)\n    if not len(known) or not len(positives):\n        return float("inf"), float("nan"), float("nan")\n    # Strict acceptance (< threshold) makes the measured FPR explicit and\n    # avoids silently exceeding the requested operating point on ties.\n    candidates = np.unique(np.concatenate(([known.max()], known)))\n    best = (float("inf"), 0.0, float("inf"))\n    for threshold in candidates:\n        fpr = float(np.mean(known >= threshold))\n        recall = float(np.mean(positives >= threshold))\n        if fpr <= target + 1e-12 and (recall > best[1] or (recall == best[1] and threshold < best[2])):\n            best = (float(threshold), recall, fpr)\n    if not np.isfinite(best[2]):\n        threshold = float(np.nextafter(known.max(), np.inf))\n        return threshold, 0.0, 0.0\n    return best\n\n\ndef classifier_accuracy(pred: np.ndarray, true_known_class: np.ndarray, y_novel: np.ndarray, accepted: np.ndarray) -> float:\n    if not bool(accepted.any()):\n        return 0.0\n    correct = (y_novel[accepted] == 0) & (pred[accepted] == true_known_class[accepted])\n    return float(np.mean(correct))\n\n\ndef risk_coverage(scores: np.ndarray, pred: np.ndarray, true_known_class: np.ndarray, y_novel: np.ndarray) -> list[dict[str, float | int]]:\n    rows: list[dict[str, float | int]] = []\n    for coverage_target in np.linspace(0.05, 1.0, 20):\n        threshold = float(np.quantile(scores, coverage_target, method="linear"))\n        accepted = scores < threshold\n        if not bool(accepted.any()):\n            accepted[np.argmin(scores)] = True\n        accuracy = classifier_accuracy(pred, true_known_class, y_novel, accepted)\n        rows.append({"threshold": threshold, "coverage": float(np.mean(accepted)), "risk": float(1.0 - accuracy) if np.isfinite(accuracy) else 1.0, "n_accepted": int(accepted.sum()), "accuracy": accuracy})\n    return rows\n\n\ndef detector_metrics(name: str, scores: np.ndarray, y_novel: np.ndarray, pred: np.ndarray, true_known_class: np.ndarray) -> dict[str, Any]:\n    threshold, recall, fpr = threshold_at_fpr(scores, y_novel)\n    accepted = scores < threshold\n    return {\n        "detector": name,\n        "auroc": auroc(y_novel, scores),\n        "threshold_at_5pct_fpr": threshold,\n        "measured_fpr": fpr,\n        "recall_at_5pct_fpr": recall,\n        "accepted_accuracy": classifier_accuracy(pred, true_known_class, y_novel, accepted),\n        "accepted_coverage": float(np.mean(accepted)),\n        "accepted_n": int(accepted.sum()),\n        "risk_coverage": risk_coverage(scores, pred, true_known_class, y_novel),\n    }\n\n\ndef train_family(family: str, df: Any, X: np.ndarray, out: Path, args: argparse.Namespace, device: Any, torch: Any, timm: Any) -> dict[str, Any]:\n    result_path = out / f"lofo_{family}.json"\n    if result_path.exists():\n        print(f"SKIP {family}: {result_path.name} already exists", flush=True)\n        return json.loads(result_path.read_text(encoding="utf-8"))\n    classes = sorted(str(x) for x in df["cls"].unique())\n    known_classes = [c for c in classes if c != family]\n    if len(known_classes) != 4:\n        raise ValueError(f"expected five classes including {family}, got {classes}")\n    split = df["split"].astype(str).str.lower().to_numpy()\n    folds = df["fold"].astype(int).to_numpy()\n    cls = df["cls"].astype(str).to_numpy()\n    train_idx = np.flatnonzero((split == "dev") & (folds != 0) & (cls != family))\n    eval_idx = np.flatnonzero(((split == "dev") & (folds == 0)) | (split == "cond_holdout"))\n    if not len(train_idx) or not len(eval_idx):\n        raise ValueError(f"{family} train/eval rows are empty: {len(train_idx)}/{len(eval_idx)}")\n    class_to_id = {name: i for i, name in enumerate(known_classes)}\n    y_train = np.asarray([class_to_id[str(cls[i])] for i in train_idx], dtype=np.int64)\n    y_eval_known = np.asarray([class_to_id.get(str(cls[i]), -1) for i in eval_idx], dtype=np.int64)\n    y_novel = (cls[eval_idx] == family).astype(np.int64)\n    checkpoint_path = out / f"lofo_{family}.pt"\n    model = make_model(timm, device, len(known_classes), pretrained=not args.dry_run)\n    counts = np.bincount(y_train, minlength=len(known_classes)).astype(np.float64)\n    weights = (counts.sum() / np.maximum(counts, 1)) ** 0.5\n    weights = torch.tensor(weights / weights.mean(), dtype=torch.float32, device=device)\n    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)\n    steps = max(1, math.ceil(len(train_idx) / args.batch_size))\n    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=max(1, args.epochs * steps), pct_start=0.15)\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    start_epoch = 0\n    epoch_logs: list[dict[str, float]] = []\n    if checkpoint_path.exists():\n        try:\n            state = torch.load(checkpoint_path, map_location=device)\n            model.load_state_dict(state["model"])\n            optimizer.load_state_dict(state["optimizer"])\n            scheduler.load_state_dict(state["scheduler"])\n            scaler.load_state_dict(state.get("scaler", {}))\n            start_epoch = int(state.get("epoch", 0))\n            epoch_logs = list(state.get("epoch_logs", []))\n            print(f"RESUME {family} from epoch={start_epoch}", flush=True)\n        except Exception as exc:\n            print(f"checkpoint unreadable for {family}; restarting ({type(exc).__name__}: {exc})", flush=True)\n    train_rng = np.random.default_rng(args.seed + FAMILIES.index(family))\n    started = time.perf_counter()\n    for epoch in range(start_epoch, args.epochs):\n        model.train()\n        order = train_rng.permutation(train_idx)\n        loss_total = 0.0\n        seen = 0\n        t0 = time.perf_counter()\n        for start_i in range(0, len(order), args.batch_size):\n            ids = order[start_i:start_i + args.batch_size]\n            xb = make_input(torch, X[ids], device)\n            xb = augment_gpu(torch, xb, args.seed + epoch * 1000 + start_i, args.img)\n            yb = torch.from_numpy(np.asarray([class_to_id[str(cls[i])] for i in ids], dtype=np.int64)).to(device=device)\n            optimizer.zero_grad(set_to_none=True)\n            with amp_context(torch, device):\n                logits, _ = forward_with_embedding(model, xb)\n                loss = loss_fn(logits, yb)\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            loss_total += float(loss.detach().cpu())\n            seen += len(ids)\n        img_s = seen / max(1e-9, time.perf_counter() - t0)\n        log = {"epoch": float(epoch + 1), "loss": float(loss_total / max(1, steps)), "img_s": float(img_s)}\n        epoch_logs.append(log)\n        print(f"family={family} epoch={epoch + 1}/{args.epochs} loss={log[\'loss\']:.5f} img/s={img_s:.2f}", flush=True)\n        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch + 1, "epoch_logs": epoch_logs, "family": family, "classes": known_classes, "script_sha256": script_sha256()}, checkpoint_path)\n    train_logits, train_emb = predict(model, X, train_idx, device, args.batch_size, torch)\n    eval_logits, eval_emb = predict(model, X, eval_idx, device, args.batch_size, torch)\n    del train_logits\n    probs = np.exp(eval_logits - eval_logits.max(axis=1, keepdims=True))\n    probs /= np.maximum(probs.sum(axis=1, keepdims=True), 1e-12)\n    max_softmax = 1.0 - probs.max(axis=1)\n    energy = -np.logaddexp.reduce(eval_logits, axis=1)\n    bank = normalize_rows(train_emb)\n    query = normalize_rows(eval_emb)\n    similarities = query @ bank.T\n    k = min(5, similarities.shape[1])\n    knn_cosine = 1.0 - np.mean(np.partition(similarities, -k, axis=1)[:, -k:], axis=1)\n    pred = eval_logits.argmax(axis=1)\n    detectors = [detector_metrics("one_minus_max_softmax", max_softmax, y_novel, pred, y_eval_known), detector_metrics("energy", energy, y_novel, pred, y_eval_known), detector_metrics("knn_cosine_distance_k5", knn_cosine, y_novel, pred, y_eval_known)]\n    unit_ids = [Path(str(x)).stem for x in df.iloc[eval_idx]["file"].tolist()]\n    split_counts = {str(key): int(value) for key, value in zip(*np.unique(split[eval_idx], return_counts=True))}\n    result: dict[str, Any] = {"family": family, "held_out_family": family, "known_classes": known_classes, "detectors": detectors, "eval": {"n": int(len(eval_idx)), "known_n": int(np.sum(y_novel == 0)), "novel_n": int(np.sum(y_novel == 1)), "split_counts": split_counts, "unit_ids": unit_ids}, "train_n": int(len(train_idx)), "epochs": epoch_logs, "wall_seconds": float(time.perf_counter() - started), "img_s_mean": float(np.mean([item["img_s"] for item in epoch_logs])) if epoch_logs else float("nan"), "script_sha256": script_sha256(), "assumptions": {"novelty_label": "family F is treated as novel against known classes; association, not proven causation", "data_scope": "fold-0 val + cond_holdout, with locked holdout excluded"}}\n    result_path.write_text(json.dumps(result, indent=2, allow_nan=False), encoding="utf-8")\n    print(f"DONE {family}: train_n={len(train_idx)} eval_n={len(eval_idx)} wall_seconds={result[\'wall_seconds\']:.2f}", flush=True)\n    return result\n\n\ndef aggregate(out: Path, families: list[str]) -> dict[str, Any]:\n    results = {family: json.loads((out / f"lofo_{family}.json").read_text(encoding="utf-8")) for family in families}\n    summary: dict[str, Any] = {}\n    for detector in ("one_minus_max_softmax", "energy", "knn_cosine_distance_k5"):\n        values = [float(next(row for row in result["detectors"] if row["detector"] == detector)["auroc"]) for result in results.values()]\n        summary[detector] = {"mean_auroc": float(np.mean(values)), "min_auroc": float(np.min(values)), "max_auroc": float(np.max(values)), "n_families": len(values)}\n    payload = {"families": results, "summary": summary, "families_completed": len(results), "families_requested": families, "script_sha256": script_sha256(), "assumptions": {"interpretation": "association, not proven causation", "scope": "locked holdout excluded"}}\n    (out / "novelty_lofo.json").write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")\n    return payload\n\n\ndef launch_two_gpus(args: argparse.Namespace) -> int:\n    script = str(Path(__file__).resolve())\n    requested = [x.strip() for x in args.families.split(",") if x.strip()]\n    groups = [requested[:2], requested[2:]]\n    processes: list[tuple[str, Any]] = []\n    for gpu, group in enumerate(groups):\n        if not group:\n            continue\n        env = os.environ.copy()\n        env["CUDA_VISIBLE_DEVICES"] = str(gpu)\n        command = [sys.executable, script, "--data-root", args.data_root, "--splits-csv", args.splits_csv, "--out", args.out, "--families", ",".join(group), "--epochs", str(args.epochs), "--img", str(args.img), "--batch-size", str(args.batch_size), "--gpu", "0"]\n        if args.dry_run:\n            command.append("--dry-run")\n        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        processes.append((f"gpu{gpu}", proc))\n\n    def tail(label: str, proc: Any) -> None:\n        assert proc.stdout is not None\n        for line in proc.stdout:\n            print(f"[{label}] {line}", end="", flush=True)\n\n    threads = [threading.Thread(target=tail, args=item, daemon=True) for item in processes]\n    for thread in threads:\n        thread.start()\n    codes = [proc.wait() for _, proc in processes]\n    for thread in threads:\n        thread.join(timeout=2)\n    if max(codes, default=0) == 0:\n        aggregate(Path(args.out), requested)\n    return max(codes, default=0)\n\n\ndef main(argv: Sequence[str] | None = None) -> int:\n    args = parse_args(argv)\n    requested = [x.strip().lower() for x in args.families.split(",") if x.strip()]\n    unknown = sorted(set(requested).difference(FAMILIES))\n    if unknown:\n        raise ValueError(f"unknown family names: {unknown}")\n    if args.launch_two_gpus:\n        return launch_two_gpus(args)\n    if args.dry_run:\n        args.epochs = 1\n        args.img = min(args.img, 64)\n        args.batch_size = min(args.batch_size, 16)\n    seed_everything(args.seed)\n    try:\n        import torch\n        import timm\n    except Exception as exc:\n        raise RuntimeError("torch, timm, pandas, and Pillow are required; no packages were installed by this script") from exc\n    if args.gpu.lower() in {"cpu", "-1", "none"} or not torch.cuda.is_available():\n        device = torch.device("cpu")\n    else:\n        device = torch.device(f"cuda:{args.gpu}" if args.gpu.isdigit() else "cuda")\n    print_banner(str(device))\n    split_path = find_splits_csv(args.splits_csv)\n    df = read_metadata(split_path)\n    classes = sorted(str(x) for x in df["cls"].unique())\n    if sorted(classes) != sorted(["crack", "hole", "normal", "rust", "scratch"]):\n        raise ValueError(f"expected crack/hole/normal/rust/scratch, got {classes}")\n    if args.dry_run:\n        # Keep all folds represented while making the local CPU test tiny.\n        split = df["split"].astype(str).str.lower().to_numpy()\n        folds = df["fold"].astype(int).to_numpy()\n        cls = df["cls"].astype(str).to_numpy()\n        rng = np.random.default_rng(args.seed)\n        chosen: list[int] = []\n        for name in classes:\n            for fold in range(5):\n                candidates = np.flatnonzero((split == "dev") & (folds == fold) & (cls == name))\n                chosen.extend(rng.choice(candidates, size=min(2, len(candidates)), replace=False).tolist())\n            candidates = np.flatnonzero((split == "cond_holdout") & (cls == name))\n            chosen.extend(rng.choice(candidates, size=min(2, len(candidates)), replace=False).tolist())\n        df = df.iloc[np.unique(chosen)].reset_index(drop=True)\n        print(f"DRY_RUN subset: {len(df)} safe rows", flush=True)\n    root = find_data_root(args.data_root, split_path, str(df["file"].iloc[0]))\n    X = load_images(df, root, args.img)\n    out = Path(args.out)\n    out.mkdir(parents=True, exist_ok=True)\n    for family in requested:\n        train_family(family, df, X, out, args, device, torch, timm)\n    aggregate(out, requested)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
Path('/kaggle/working/lofo_novelty.py').write_text(SOURCE, encoding='utf-8')
print('wrote /kaggle/working/lofo_novelty.py')
print('script_sha256_actual', __import__('hashlib').sha256(SOURCE.encode('utf-8')).hexdigest())

In [ ]:
DRY_RUN = True
import sys
sys.path.insert(0, '/kaggle/working')
from lofo_novelty import main
args = ['--splits-csv', '/kaggle/working/splits.csv', '--out', '/kaggle/working', '--families', 'crack,hole,rust,scratch', '--gpu', '0']
if DRY_RUN:
    args.append('--dry-run')
else:
    args.append('--launch-two-gpus')
print('launching:', args, 'DRY_RUN=', DRY_RUN)
raise SystemExit(main(args))
